# Notebook 03 — Writing Introductions

**Module:** 18 — Scientific Writing and Open Science  
**Tier:** 2 — Working competence  
**Notebook:** 3 of 12  
**Time estimate:** 75 minutes

> The introduction must perform one job: convince the reader that there is a
> real problem that has not yet been solved, and that this paper addresses it.
> Every sentence serves that argument or is cut.

---
## Step 1 — Motivation

The introduction is the part that editors read first when deciding whether to
send a paper for review. It is the part that reviewers read to frame everything
that follows. A weak introduction — one that buries the question, inflates
background, or fails to state the gap — is the leading cause of desk rejection.

---
## Step 2 — The Four-Paragraph Introduction

For a computational biology methods or analysis paper, 4 paragraphs is the target.

| Paragraph | Content | Length | Key rule |
|-----------|---------|--------|----------|
| **P1: Field and importance** | Why does the broad problem matter? | 4–6 sentences | Open with a claim, not a definition. End with the specific sub-problem. |
| **P2: State of the art** | What have others done? What works? | 5–8 sentences | Cite key papers. Be fair — don't dismiss competitors unfairly. |
| **P3: The gap / limitation** | What is still unsolved? Why does it matter? | 3–5 sentences | This is the most important paragraph. Make the gap crisp and specific. |
| **P4: This paper** | What do you do? What do you show? | 3–5 sentences | "Here we present/develop/apply X. We show that... We demonstrate..." |

**Funnel structure:** P1 is broad → P2 narrows → P3 is the specific gap → P4 is your paper.
The funnel is not metaphorical. Word count should shrink as the paragraphs progress.

**P4 must not be vague.** Avoid: "We develop a novel approach." Use instead:
"Here we present [Name], a convolutional neural network trained on X datasets from
[source], that achieves Y% improvement over the PWM baseline on the Z benchmark.
We show that learned filters correspond to known binding motifs (Figure 3) and
that [Name] generalizes to held-out TFs (Figure 5)."

---
## Step 3 — Active Voice and Strong Verbs

| Weak (passive) | Strong (active) |
|---------------|----------------|
| "A dataset was assembled from ENCODE" | "We assembled a dataset from ENCODE" |
| "Sequences were predicted using DeepBind" | "DeepBind predicted sequences" |
| "It was found that accuracy improved" | "Accuracy improved from 0.81 to 0.92" |

Verb choice matters too. Prefer: *reveal, demonstrate, achieve, establish, show,
identify, develop, present, uncover.* Avoid: *utilize* (use "use"), *perform*
(use "do"), *utilize* (always use "use"), *leverage* ("use"), *novel* (show it, don't claim it).

---
## Step 4 — The Gap Statement

The gap statement (P3) has four sub-parts:
1. **What current methods do** — brief, 1 sentence
2. **The specific failure mode** — what they get wrong, and in what conditions
3. **Why it matters** — biological or clinical consequence of this failure
4. **What would need to be different** — points to your approach without naming it yet

Bad gap: "However, current methods have some limitations."
Good gap: "Position weight matrix methods treat binding positions as independent,
failing to model the cooperativity between adjacent nucleotides that governs binding
specificity in >60% of experimentally characterized TFs (CITE). This leads to a
13% increase in false positives at experimentally validated binding sites (CITE).
An approach that models position interactions directly would improve predictive accuracy."

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# ---- Introduction analysis tools ----

def paragraph_word_counts(text):
    """Split text into paragraphs and count words per paragraph."""
    paragraphs = [p.strip() for p in text.strip().split('\n\n') if len(p.strip()) > 20]
    return [(i+1, len(p.split()), p[:60] + '...') for i, p in enumerate(paragraphs)]

def verb_strength_audit(text):
    """Flag weak verbs and count strong verbs."""
    weak_verbs = ['utilize', 'perform', 'leverage', 'implement', 'employ',
                    'conduct', 'undertake', 'novel', 'innovative', 'demonstrate the potential']
    strong_verbs = ['show', 'reveal', 'achieve', 'establish', 'identify', 'develop',
                      'present', 'uncover', 'demonstrate', 'find', 'outperform']
    text_lower = text.lower()
    weak_found = [v for v in weak_verbs if v in text_lower]
    strong_count = sum(1 for v in strong_verbs if v in text_lower)
    return weak_found, strong_count

def gap_signal_strength(paragraph):
    """Score how clearly a paragraph signals a research gap."""
    gap_signals = ['however', 'despite', 'yet', 'limitation', 'fail', 'cannot',
                     'does not', 'unable', 'missing', 'overlooked', 'challenge',
                     'remains', 'unclear', 'unresolved']
    p_lower = paragraph.lower()
    score = sum(1 for g in gap_signals if g in p_lower)
    return score

def passive_sentences(text):
    """Return list of sentences likely in passive voice."""
    pattern = r'\b(was|were|been|being|is|are)\s+\w+ed\b'
    sents = [s.strip() for s in re.split(r'[.!?]', text) if len(s.strip()) > 10]
    return [s for s in sents if re.search(pattern, s, re.IGNORECASE)]

# ---- Synthetic introduction example ----

INTRO_WEAK = """
Transcription factors are proteins that bind to DNA and control gene expression.
Gene expression is important. Many transcription factors have been studied.
Researchers have developed many methods to study them.

Position weight matrices (PWMs) are one approach. PWMs were introduced by Stormo et al.
Many tools use PWMs. FIMO, MEME, and Jaspar all use PWMs. They work reasonably well
in many cases but sometimes fail.

There are limitations to existing approaches. PWMs have some problems. It would be
useful to have a new method.

In this paper, we develop a new approach. We show it works well. The code is available.
"""

INTRO_STRONG = """
Transcription factors (TFs) orchestrate gene expression programs in development, differentiation,
and disease. Misregulation of TF binding underlies oncogenesis in many tumour types and drives
congenital developmental disorders. Identifying the genomic sequences that TFs recognize is
therefore fundamental to understanding the molecular logic of gene regulation and to designing
therapeutic interventions that target transcriptional networks.

Position weight matrices (PWMs) have been the dominant tool for TF binding site prediction
since their introduction by Stormo and colleagues in 1982. Tools including FIMO, MEME, and
the JASPAR database give researchers access to curated PWM libraries for over 700 human TFs.
Recent deep learning methods have demonstrated improved performance by learning sequence
features from large ChIP-seq datasets.

However, PWMs treat each nucleotide position as statistically independent, failing to model
the cooperative interactions between adjacent residues that govern binding specificity in the
majority of TFs studied to date. This independence assumption leads to a systematic inflation
of predicted binding sites: at an empirically validated set of 50,000 ChIP-seq peaks,
PWM-based methods produce approximately 13% more false positives than methods that model
positional dependencies.

Here we present DeepBind-v2, a convolutional neural network trained end-to-end on ChIP-seq
data from 690 human TFs in ENCODE. DeepBind-v2 learns sequence features without relying
on pre-specified motif models, achieving a mean AUROC of 0.924 across all TFs and
outperforming the best PWM-based baseline by 12.3 percentage points. We show that learned
convolutional filters recover known TF binding motifs, that the model generalizes to held-out
TFs not seen during training, and that filter activations provide interpretable evidence for
cooperative binding. DeepBind-v2 is freely available at https://github.com/example/deepbind-v2.
"""

# Audit both
for name, intro in [('WEAK', INTRO_WEAK), ('STRONG', INTRO_STRONG)]:
    print(f'\n=== {name} introduction ===')
    para_wc = paragraph_word_counts(intro)
    for p_num, wc, preview in para_wc:
        print(f'  P{p_num} ({wc} words): {preview}')
    weak_v, strong_v = verb_strength_audit(intro)
    print(f'  Weak verbs found: {weak_v if weak_v else "none"}')
    print(f'  Strong verb count: {strong_v}')
    paras = [p.strip() for p in intro.strip().split('\n\n') if len(p.strip()) > 20]
    if len(paras) >= 3:
        print(f'  Gap signal score (P3): {gap_signal_strength(paras[2])}')
    passive = passive_sentences(intro)
    print(f'  Passive voice sentences: {len(passive)}')

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# Panel A: Funnel structure diagram
ax = axes[0]
ax.axis('off')
ax.set_xlim(0, 10); ax.set_ylim(0, 10)

# Draw funnel as trapezoids narrowing downward
funnel_data = [
    (1, 9, 8, 1.4, 'P1: Field and importance', '#4e79a7', 'Broad: why this matters'),
    (2, 7.2, 6, 1.4, 'P2: State of the art', '#f28e2b', 'Narrowing: what exists'),
    (3, 5.4, 4, 1.4, 'P3: The gap', '#e15759', 'Specific: what is missing'),
    (4, 3.6, 2, 1.4, 'P4: This paper', '#59a14f', 'Narrow: what you do'),
]
for left, y, width, height, label, color, desc in funnel_data:
    from matplotlib.patches import FancyBboxPatch
    ax.add_patch(FancyBboxPatch((left, y), width, height,
                                  boxstyle='round,pad=0.1',
                                  facecolor=color, alpha=0.7, edgecolor='black'))
    ax.text(left + width/2, y + height/2 + 0.1, label, ha='center', va='center',
              fontsize=9, fontweight='bold')
    ax.text(left + width/2, y + height/2 - 0.3, desc, ha='center', va='center', fontsize=8)
ax.set_title('A. Introduction funnel structure\n(broad → specific → this paper)', fontsize=11)

# Panel B: Paragraph word counts comparison
ax = axes[1]
weak_wc  = [len(p.split()) for p in INTRO_WEAK.strip().split('\n\n') if len(p.strip()) > 20]
strong_wc = [len(p.split()) for p in INTRO_STRONG.strip().split('\n\n') if len(p.strip()) > 20]
x = np.arange(4)
w = 0.35
ax.bar(x - w/2, weak_wc[:4],   w, label='Weak',   color='tomato',    edgecolor='black', alpha=0.8)
ax.bar(x + w/2, strong_wc[:4], w, label='Strong', color='steelblue', edgecolor='black', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(['P1 Field', 'P2 State\nof art', 'P3 Gap', 'P4 This\npaper'])
ax.set_ylabel('Word count')
ax.set_title('B. Paragraph word counts\n(strong intro has decreasing funnel)')
ax.legend(fontsize=9)

# Panel C: Gap signal strength comparison
ax = axes[2]
gap_signals = ['however', 'despite', 'yet', 'limitation', 'fail', 'cannot',
                 'does not', 'unable', 'missing', 'challenge', 'remains', 'unresolved']

def count_gap_signals_per_paragraph(intro):
    paras = [p.strip() for p in intro.strip().split('\n\n') if len(p.strip()) > 20]
    return [sum(1 for g in gap_signals if g in p.lower()) for p in paras]

weak_gaps   = count_gap_signals_per_paragraph(INTRO_WEAK)
strong_gaps = count_gap_signals_per_paragraph(INTRO_STRONG)

x = np.arange(4)
ax.bar(x - w/2, weak_gaps[:4],   w, label='Weak',   color='tomato',    edgecolor='black', alpha=0.8)
ax.bar(x + w/2, strong_gaps[:4], w, label='Strong', color='steelblue', edgecolor='black', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(['P1 Field', 'P2 State\nof art', 'P3 Gap', 'P4 This\npaper'])
ax.set_ylabel('Gap signal word count')
ax.set_title('C. Gap signal by paragraph\n(should peak at P3, not P2 or P4)')
ax.legend(fontsize=9)

plt.suptitle('Module 18 NB03: Writing Introductions', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('writing_introductions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

See E03 in `exercises/README.md`: diagnose the weak introduction above —
list every problem by paragraph, then rewrite P3 (the gap paragraph) to
match the strong-introduction standard.

---
## Step 10 — Quiz

1. What is the funnel structure of an introduction? Why does it narrow?
2. Write a one-sentence P3 gap statement for: "Existing RNA-seq DE methods
   assume negative-binomial distributions, but single-cell data is zero-inflated."
   Include (a) what current methods do wrong, (b) why it matters, (c) what
   needs to change.
3. What is the difference between P4 in a weak vs. strong introduction?
   Give a concrete example of each for a genomics classifier paper.
4. Name three weak verbs to avoid and three strong verbs to prefer.

---
## Step 12 — Reflection

> *[Write P3 (the gap paragraph) for MP01 — your chosen prior result. Use the
> four-subpart structure: what current methods do, their specific failure mode,
> why it matters, what would need to be different. Target: 60–80 words.]*

---
*Next: `04_methods_and_reproducibility.ipynb`*